To load base model as well as lora parameters finetuned by SFTT, which will be tested by inference with specific inputs.  

In [1]:
%%capture
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1" # [NEW] Extra 30% context lengths!
!pip install --upgrade -qqq uv
try: import numpy, PIL; get_numpy = f"numpy=={numpy.__version__}"; get_pil = f"pillow=={PIL.__version__}"
except: get_numpy = "numpy"; get_pil = "pillow"
try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
except: is_t4 = False
get_vllm, get_triton = ("vllm==0.8.5", "triton==3.2.0") 
!uv pip install -qqq --upgrade     unsloth {get_vllm} {get_numpy} {get_pil} torchvision bitsandbytes xformers
!uv pip install -qqq {get_triton}
!uv pip install "huggingface_hub>=0.34.0" "datasets>=3.4.1,<4.0.
!uv pip install transformers==4.53.2
!uv pip install torch==2.6.0
!uv pip install --no-deps trl==0.22.2

In [2]:
#!uv pip install unsloth==2025.7.5
!uv pip install vllm==0.8.5.post1
!uv pip install xformers==0.0.29.post3
!uv pip install triton==3.2.0
!uv pip install bitsandbytes

Using Python 3.11.13 environment at: /usr
Resolved 161 packages in 131ms
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠹ Preparing packages... (0/1)
⠹ Preparing packages... (0/1)
⠹ Preparing packages... (0/1)
⠹ Preparing packages... (0/1)
⠸ Preparing packages... (0/1)
⠸ Preparing packages... (0/1)
⠸ Preparing packages... (0/1)
⠼ Preparing packages... (0/1)
⠼ Preparing packages... (0/1)
⠼ Preparing packages... (0/

In [3]:
# Install other dependencies
!uv pip install "huggingface_hub>=0.34.0" "datasets>=3.4.1,<4.0.0"
!uv pip install transformers==4.53.2
!uv pip install torch==2.6.0
!uv pip install --no-deps trl==0.22.2
!uv pip install torchvision

Using Python 3.11.13 environment at: /usr
Resolved 45 packages in 73ms
⠙ Preparing packages... (0/3)
⠙ Preparing packages... (0/3)
⠙ Preparing packages... (0/3)
dill                 ------------------------------     0 B/113.53 KiB
⠙ Preparing packages... (0/3)
dill                 ------------------------------     0 B/113.53 KiB
⠙ Preparing packages... (0/3)
dill                 ------------------------------     0 B/113.53 KiB
⠙ Preparing packages... (0/3)
dill                 ------------------------------     0 B/113.53 KiB
fsspec               ------------------------------     0 B/189.08 KiB
⠙ Preparing packages... (0/3)
dill                 ------------------------------ 16.00 KiB/113.53 KiB
fsspec               ------------------------------     0 B/189.08 KiB
⠙ Preparing packages... (0/3)
dill                 ------------------------------ 32.00 KiB/113.53 KiB
fsspec               ------------------------------     0 B/189.08 KiB
⠙ Preparing packages... (0/3)
dill            

In [4]:
# --- 1. Configuration ---
import os
import pandas as pd
import torch
import re
import io
import sys
import ast
import time
from transformers import AutoTokenizer
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [5]:
if torch.cuda.is_available():
    device = torch.device("cuda:0")
    print(f"Using device: {device}")
else:
    device = torch.device("cpu")
    print("CUDA is not available. Using CPU.")

Using device: cuda:0


In [6]:
# --- 2. Load finetuned Model and Tokenizer ---

# Define paths for the base model and the LoRA adapter
base_model_path = "/kaggle/input/qwen3-4b-sfft-merged/transformers/default/1/Qwen3-4B-SFFT-merged"
#lora_adapter_path = "/kaggle/input/lora-2-3/sftt_save_lora_v1_3"

In [7]:
from unsloth import FastLanguageModel
import torch
from peft import PeftModel
max_seq_length = 2048 # Can increase for longer reasoning traces
lora_rank = 32 # Larger rank = smarter, but slower
# --- Load base model ---

base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = base_model_path,
    max_seq_length = max_seq_length,
    load_in_4bit = False, # False for LoRA 16bit
    torch_dtype = torch.float16, # Force float16 for T4
    #fast_inference = True, # Enable vLLM fast inference
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.9, # Reduce if out of memory
)
print("Merged model loaded successfully with Unsloth.")   
  

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/tmp/ipykernel_19/1816185404.py:1: UserWarning: WARNING: Unsloth should be imported before transformers to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel
2025-11-19 09:02:17.620576: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763542937.835903      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763542937.897838      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


INFO 11-19 09:02:47 [importing.py:53] Triton module has been replaced with a placeholder.
INFO 11-19 09:02:47 [__init__.py:239] Automatically detected platform cuda.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.11.3: Fast Qwen3 patching. Transformers: 4.53.2. vLLM: 0.8.5.post1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Merged model loaded successfully with Unsloth.


In [8]:
model = base_model

In [9]:
# --- VERIFICATION ---
print("\n--- Tokenizer and Model Verification ---")
print(f"Tokenizer vocab size: {len(tokenizer)}")
print(f"Model embedding layer size: {model.get_input_embeddings().weight.size(0)}")
print(f"Is tokenizer vocab size == model embedding size: {len(tokenizer) == model.get_input_embeddings().weight.size(0)}")
print(f"Loaded Pad Token: {tokenizer.pad_token}")
print("--------------------------------------\n")


--- Tokenizer and Model Verification ---
Tokenizer vocab size: 151669
Model embedding layer size: 151936
Is tokenizer vocab size == model embedding size: False
Loaded Pad Token: <|endoftext|>
--------------------------------------



In [10]:
# --- 2. SYSTEM PROMPTS (Must match training) ---
code_system_prompt = \
"""You are an expert Python programmer. Your sole task is to write a self-contained Python script to solve the given computational problem.
- Your response MUST begin directly with the code block ```python and end with ```.
- Do NOT provide any text or explanation before or after the code block.
- The script must define a function `solve()` that returns the final numerical answer.
- The script must then call the `solve()` function. The result should be the final expression of the script.
- Do NOT solve the problem yourself or provide any reasoning inside the script, just return the caculation.
- Analyze the problem carefully and choose the appropriate response format, which should contain non-repetitive answers.
- For example, for 'caculate the value of (1+1)', your entire response must be: '```python
def solve():
    return 1+1
print(solve())
```'.
"""

cot_code_system_prompt = \
"""You are a multi-talented expert in mathematics and Python programming. Your task is to solve the given problem by providing both a textual explanation and a Python script.
- First, provide a clear, step-by-step explanation of your reasoning.
- After the explanation, provide a complete, self-contained Python script inside a ```python ... ``` block.
- The script should define a function `solve()` that returns the final numerical answer.
- The script must then call the `solve()` function. The result should be the final expression of the script.
- For example, for 'caculate the value of (1+1)', your script need to be: '```python
def solve():
    return 1+1
print(solve())
```'.
"""

In [11]:
tokenizer.chat_template

"{% if messages[0]['role'] == 'system' %}{{ messages[0]['content'] + eos_token }}{% set loop_messages = messages[1:] %}{% else %}{{ 'You are given a problem.\nThink about the problem and provide your working out.\nPlace it between <start_working_out> and <end_working_out>.\nThen, provide your solution between <SOLUTION></SOLUTION>' + eos_token }}{% set loop_messages = messages %}{% endif %}{% for message in loop_messages %}{% if message['role'] == 'user' %}{{ message['content'] }}{% elif message['role'] == 'assistant' %}{{ message['content'] + eos_token }}{% endif %}{% endfor %}{% if add_generation_prompt %}{{ '<start_working_out>' }}{% endif %}"

In [12]:
# --- 2.1 To determine which type of problem it is and then encapsulate it with respective prompt  
def select_prompt(problem: str) -> str:
    """
    Selects the appropriate system prompt based on the problem's content.
    This should roughly match the logic used to categorize data during training.
    """
    problem_lower = problem.lower()

    if "calculate the" in problem_lower:
        print("(System 01: Detected computational problem, using Code-Only prompt.)")
        #problem = problem_lower.replace("calculate","$Calculate#")
        problem = problem.replace("Calculate","$Calculate#")
        return code_system_prompt, problem
    
    # Keywords that suggest a need for reasoning/proof (CoT)
    proof_keywords = [
        # Original keywords
        'prove that', 'show that', 'demonstrate that', 'explain why',
        'is it true that', 'determine', 'converge', 'convergent',
        'relationship', 'what is the probability',
        # Added keywords from user examples and common math terms
        'equation', 'derivative', 'expansion', 'how many', 'what is',
        'compute the', 'simplify', 'solve for', 'express', 'theorem',
        'proof', 'show', 'derive', 'relates', 'find an', 'find the', 'calculate the'
    ]
    
    # Mathematical symbols/patterns that often appear in theoretical problems
    proof_symbols = [
        # Original symbols
        r'\\sum', r'\\int', r'\\lim', r'\\infty', r'\\binom', r'\\choose',
        r'\\prod', r'\\partial', r'\\nabla', r'\\subset', r'\\in',
        # Added symbols and patterns
        r'\\theta', r'\\sec', r'\\pi', r'\\alpha', r'\\beta', r'\\gamma',
        r'\\delta', r'\\sin', r'\\cos', r'\\tan', r'\\log',
        # Match 'ln' as a word, or '\\ln' for latex
        r'\\ln', r'\bln\b',
        r'\\sqrt',
        # Match f(x), g(x), etc.
        r'f\(x\)', r'g\(x\)', r'h\(x\)',
        r'd/dx', r'\^',
        # Corrected patterns for escaped parentheses for LaTeX e.g. \\( ... \\)
        r'\\\(' , r'\\\)',
        # Pattern for simple parentheses in computational problems e.g. (1+2)
        # This is a broad match, but problems with parens are often not simple arithmetic.
        r'\('
    ]

    if any(keyword in problem_lower for keyword in proof_keywords) or \
       any(re.search(symbol, problem) for symbol in proof_symbols):
        print("(System 02: Detected theoretical problem, using CoT+Code prompt.)")
        return cot_code_system_prompt, problem
    else:
        print("(System 03: Detected computational problem, using Code-Only prompt.)")
        return code_system_prompt, problem

In [13]:
# --- 2.2. Prepare the test questions set ---
question_set=["Calculate the value of (0x1235 XOR 0xfb67)",
              "Calculate the value of (0x36789 AND 0x67fac OR 0x209b7)",
              "Calculate the value of (1567*12 + 3**17 + ln(7))",
              "Calculate the value of (56129087+23458765)",
              "Calculate the value of (87165*33921)",
              "Calculate the value of (2765420987-127654987)",
              "Calculate the value of (76552298/7654322)",
              "Calculate the value of ((0x2a765 >> 16 ) and (0xb7 << 8))",
              "Caculate the 32-bit 0x2a765 right cyclic shift by 16 bits.",
              #"Calculate the 32-bit right cyclic shift of 0x2a765 by 16 bits.",
              #"Assume 0x2a765 is a 32-bit number, calculate the value after cyclic right-shift 16 bit ",
              "Calculate the value of (e**2 + pi*2/3 + 130)",
              "Mary has taken three tests and has a test average of 87. Her parents want her to maintain an average of at least 85. What is the lowest score that Mary can get on her next test while keeping her average at least 85?",
              "A rectangle is inscribed in a circle with an area of $16\\pi$. The area of the rectangle is $30$. Find the perimeter of the rectangle.",  
              "Given $\\sqrt{x^2+38}-\\sqrt{x^2-6}=4$ and $x$ is positive, find all possible values of $x$.",
              "What is the sum of all real numbers $x$ for which $|x^2 - 6x + 12| = 3$?",
              "Suppose that the equations $y = x^3 - 3x + 5$ and $x + 2y = 8$ intersect at the points $(x_1, y_1)$, $(x_2, y_2)$, and $(x_3, y_3)$. What is the value of $x_1 + x_2 + x_3 + y_1 + y_2 + y_3$?",
              "What is the minimum distance a fly must travel when flying from one corner to the opposite corner of a rectangular box that measures 3 feet by 5 feet by 6 feet?",
              "In a summer camp with 100 students, each student can sing, dance, or act. Some students have more than one talent, but no student has all three talents. There are 42 students who cannot sing, 65 students who cannot dance, and 29 students who cannot act. How many students have two of these talents?",
              "At his usual rowing rate, a man rows 15 miles downstream in five hours less time than it takes him to return. If he doubles his usual rowing rate, the time downstream is only one hour less than the time upstream. Find the rate of the stream's current in miles per hour.",
              "Jason has the same number of red and blue marbles. He puts them in two jars so that the ratio of red to blue marbles in jar I is 2:5 and in jar II is 9:5. If there are 84 marbles in jar I, how many marbles are there in jar II?",
              "A person is walking down an escalator moving downwards and takes a total of 26 steps to reach the bottom in 30 seconds. When running down the escalator, the same person takes a total of 34 steps and reaches the bottom in 18 seconds. How many steps are on the escalator at a time?",
              "Let $(A_{ij})$ be an $n \\times n$ matrix and $(B_{ij})$ be the cofactor matrix of $(A_{ij})$. What is the rank of $(B_{ij})$ when the rank of $(A_{ij})$ is $\\le n-2$?",
              "On a table, there are 2016 coins. Two players take turns, and in each turn, a player can take 1, 2, or 3 coins. The player who takes the last coin wins. Which player has a winning strategy?",
              "Find an equation that relates the observable distance \\(d\\) on the Earth's surface to the central angle \\(\\theta\\).",
              "Compute the derivative of \\( f(x) = e^{x \\sec(x)} \\).",
              "How many terms are in the expansion of $(a+b+c+d)^n$ (in terms of $n$)?",
              "Find the sum of the infinite series \\(9 - 3 + 1 - \\frac{1}{3} + \\frac{1}{9} + \\cdots\\).",
              "Find a formula for \\( \\binom{n}{0}^2 + \\binom{n}{1}^2 + \\binom{n}{2}^2 + \\cdots + \\binom{n}{n}^2 \\).",
              "Factor $6x^{12} + 35x^7 + 36x^2$.",
              "Study the Lebesgue measurability and integrability of the function $f:(0,1)\\rightarrow\\mathbb{R}$, which is continuous and satisfies $\\lvert f(x) \\rvert \\le \\frac{1}{\\sqrt x}$ for all $x\\in(0,1)$.",
              "Under what conditions is \\( X^T X \\) invertible?"
             ]

In [14]:
# --- 2.3. Select the question and its system prompt ---
current_question = question_set[29]
system_prompt, current_question = select_prompt(current_question)
print(current_question,"~~~~~~", system_prompt)

(System 02: Detected theoretical problem, using CoT+Code prompt.)
Under what conditions is \( X^T X \) invertible? ~~~~~~ You are a multi-talented expert in mathematics and Python programming. Your task is to solve the given problem by providing both a textual explanation and a Python script.
- First, provide a clear, step-by-step explanation of your reasoning.
- After the explanation, provide a complete, self-contained Python script inside a ```python ... ``` block.
- The script should define a function `solve()` that returns the final numerical answer.
- The script must then call the `solve()` function. The result should be the final expression of the script.
- For example, for 'caculate the value of (1+1)', your script need to be: '```python
def solve():
    return 1+1
print(solve())
```'.



In [15]:
# --- 2.4. Apply the Prompt (Crucial Step) ---
question = current_question
multi_task_system_prompt = system_prompt

# Construct the 'messages' list in the same format as your training data.
messages = [
    {"role": "system", "content": multi_task_system_prompt},
    {"role": "user", "content": question},
    #{"role": "assistant", "content": "<think></think>"},
]

# Apply the chat template to format the input correctly.
# `add_generation_prompt=True` is essential for inference.
inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True, 
        padding=True, # This is important for batching, good practice for single inference
        return_tensors="pt"
).to(device)

# --- 3. Generate the Response ---
start_time = time.time()    
prompt_token_length = inputs.shape[1]
print("\nGenerating response...")
outputs = model.generate(
    input_ids=inputs, 
    max_new_tokens=2048, 
    use_cache=True,
    do_sample=True,
    temperature=0.2, # Lower the temperature to reduce randomness
    top_p=0.9
) 

outputs_without_prompt = outputs[:, prompt_token_length: ]
response_text = tokenizer.batch_decode(outputs_without_prompt, skip_special_tokens=True)[0]
print("\nFinish reasoning.")
end_time = time.time()
ref_time = end_time - start_time
print("It cost {} seconds for reference".format(ref_time))
# --- 5. Print the Cleaned Response ---
    
# Extract only the assistant's part of the response for clarity.
try:
    assistant_response = response_text.split("<|im_start|>assistant\n")[-1].strip()
except IndexError:
    assistant_response = response_text # Fallback if the template isn't found

print("\n--- Model Response ---")
print(assistant_response)
print("----------------------\n")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



Generating response...

Finish reasoning.
It cost 113.69489407539368 seconds for reference

--- Model Response ---
To determine when the matrix \( X^T X \) is invertible, we need to analyze the properties of the matrix \( X \). Let's denote \( X \) as an \( m \times n \) matrix. The matrix \( X^T X \) is then an \( n \times n \) matrix, where \( X^T \) is the transpose of \( X \).

A matrix is invertible if and only if its determinant is non-zero. For the matrix \( X^T X \), this condition translates to the requirement that the matrix \( X^T X \) must be non-singular. A matrix is non-singular if and only if its rank is equal to the number of its rows (or columns, since it is a square matrix).

The rank of \( X^T X \) is equal to the rank of \( X \). This is because the rank of a product of matrices \( AB \) is at most the minimum of the ranks of \( A \) and \( B \). Since \( X^T X \) is a product of \( X^T \) and \( X \), the rank of \( X^T X \) is the same as the rank of \( X \).

Th

# --- 4. Extract python code from the response 
pattern = r"```(?:python|py|)\s*?(.*?)```"
matches = re.findall(pattern, assistant_response, re.DOTALL)
#print matches
#matches

if matches:
    for i, python_code in enumerate(matches):
        print(f"\nExtracted Code {i+1}:")
        print(python_code.strip())
else:
    print("No python code found.")

In [16]:
# --- 5. Extract python code from the response 
#pattern = r"```(?:python|py|)\s*?(.*?)```"
pattern = r"```+(?:python|py)?\s*\n(.*?)\n```+"
matches = re.findall(pattern, assistant_response, re.DOTALL)
#print matches
#matches

if matches:
    for i, python_code in enumerate(matches):
        print(f"\nExtracted Code {i+1}:")
        print(python_code.strip())
else:
    print("No python code found.")


Extracted Code 1:
def solve():
    # The function solve() does not need to perform any computation here.
    # The answer is simply the condition that the rank of X must be equal to n.
    return "rank(X) = n"


In [17]:
def run_code(code_string: str) -> tuple[str, bool]:
    """
    Executes a string of Python code in a safe scope and captures the output.

    This function is robust and handles three common cases for LLM-generated code:
    1. Code with explicit print() statements: It captures the printed output.
    2. Code ending in an expression (e.g., '3 + 5'): It returns the value.
    3. Code ending in a statement (e.g., 'a = 5'): It captures nothing, which is correct.

    Args:
        code_string: The Python code to execute.

    Returns:
        A tuple containing:
        - The captured output or result as a string.
        - A boolean indicating if an error occurred (True if error, False otherwise).
    """
    output_buffer = io.StringIO()
    original_stdout = sys.stdout
    scope = {}

    # Ensure stdout is restored even if errors occur
    try:
        # Redirect stdout to capture prints
        sys.stdout = output_buffer
        last_expr = None

        # Use the 'ast' module to find and evaluate the last expression
        try:
            tree = ast.parse(code_string.strip())
            if tree.body and isinstance(tree.body[-1], ast.Expr):
                # If the last node is an expression, wrap it in a print() call.
                # This makes code like 'a=5; a+10' correctly output '15'.
                last_expr_node = tree.body.pop()
                last_expr = ast.unparse(last_expr_node).strip()

                # Create the new print node
                print_node = ast.Expr(
                    value=ast.Call(
                        func=ast.Name(id='print', ctx=ast.Load()),
                        args=[last_expr_node.value],
                        keywords=[]
                    )
                )
                tree.body.append(ast.fix_missing_locations(print_node))

            # Compile the (potentially modified) code and execute it
            exec(compile(tree, '<string>', 'exec'), scope)
            #last_result = eval(last_expr, scope)
        except (SyntaxError, Exception) as e:
            # If parsing or execution fails, print the error to the real stderr
            # and also capture it in our return value.
            print(f"Error executing code: {e}", file=sys.stderr)
            return f"Error: {e}", True

    finally:
        # ALWAYS restore the original stdout
        sys.stdout = original_stdout

    # Get the complete output from the buffer and clean it up
    captured_output = output_buffer.getvalue().strip()
    if last_expr:
        if ("print" in last_expr):
            if (eval(last_expr, scope)==None):
                captured_output = output_buffer.getvalue().strip().replace("\nNone","")
    return captured_output, False

In [18]:
# --- 6 The final response with result of code execution ---

pattern = r"```+(?:python|py)?\s*\n(.*?)\n```+"
matches = re.findall(pattern, assistant_response, re.DOTALL)
revised_content = assistant_response
if matches:
    for i, code_block in enumerate(matches):
        code_block = code_block.strip()
        if not code_block:
            continue

        print(f"\n--- Executing Block {i+1} ---")
        print(code_block)

        #This one line runs the code and gets the result
        result, had_error = run_code(code_block)
            
        print(f"Result: {result}")
        print(f"Error Occurred: {had_error}")
            
        # Create the text to insert back into the model's response
        if had_error:
            # Use a distinct marker for errors
            insert_text = f"```python\n{code_block}\n\n---[Code Execution Error]---\n{result}\n```"
        else:
            # Use a clean marker for successful output
            insert_text = f"```python\n{code_block}\n\n---[Code Output]---\n{result}\n```"

        # Replace the original code block with the annotated version
        original_block_in_markdown = f"```python\n{code_block}\n```"
        revised_content = revised_content.replace(original_block_in_markdown, insert_text, 1)

print("\n\n--- Final Revised Content ---")
print(revised_content)


--- Executing Block 1 ---
def solve():
    # The function solve() does not need to perform any computation here.
    # The answer is simply the condition that the rank of X must be equal to n.
    return "rank(X) = n"
Result: 
Error Occurred: False


--- Final Revised Content ---
To determine when the matrix \( X^T X \) is invertible, we need to analyze the properties of the matrix \( X \). Let's denote \( X \) as an \( m \times n \) matrix. The matrix \( X^T X \) is then an \( n \times n \) matrix, where \( X^T \) is the transpose of \( X \).

A matrix is invertible if and only if its determinant is non-zero. For the matrix \( X^T X \), this condition translates to the requirement that the matrix \( X^T X \) must be non-singular. A matrix is non-singular if and only if its rank is equal to the number of its rows (or columns, since it is a square matrix).

The rank of \( X^T X \) is equal to the rank of \( X \). This is because the rank of a product of matrices \( AB \) is at most the

In [19]:
'''
# Release GPU resource 
import torch
from numba import cuda
torch.cuda.empty_cache()
if torch.cuda.is_available():
    print("Releasing GPU memory (cleanup)...")
    cuda.get_current_device().reset()
    print("GPU memory released.")

import gc
gc.collect()
'''

'\n# Release GPU resource \nimport torch\nfrom numba import cuda\ntorch.cuda.empty_cache()\nif torch.cuda.is_available():\n    print("Releasing GPU memory (cleanup)...")\n    cuda.get_current_device().reset()\n    print("GPU memory released.")\n\nimport gc\ngc.collect()\n'

In [20]:
# To kill the subprocess of PID directly to empty the memory of GPU in case of failure after restarting kernel 
#!kill -9 9190

In [21]:
#!nvidia-smi